# EVE Online SDE: Anomaly Detection for Game Health

## **GOAL**
To analyze the EVE Online Static Data Export (SDE) to identify statistical anomalies in game items. The primary use case is monitoring "game health" by finding items (like ships) that are statistically imbalanced in their cost-vs-stat ratios.

## **Workflow Overview**

**ETL (Extract, Transform, Load):** The raw SDE (.zip) is downloaded from the official developer link, unzipped, and all 52 .jsonl files are uploaded to a Google Cloud Storage (GCS) bucket using a shell script.

**Data Wrangling (Pandas):** The core of this project involves loading the data from GCS into pandas DataFrames. This is a complex task, as key data (stats, materials, map positions) is nested in JSON columns. We use explode() and json_normalize() to flatten this data into usable tables.

**Visualization (Plotly):** We use interactive visualizations to understand the dataset's structure:

  
  **Treemaps:** To show the item hierarchy (Categories ->  Groups -> Types).

  **Network Graphs:** To show ownership (Factions -> Corporations -> Stations).
  
  **Bar Charts:** To count items by Race.
  
  **3D Scatter Plots:** To build a fully interactive 3D map of the EVE universe.

**Feature Engineering:** We build a "master feature vector" for all Ships. This table has one row per ship and columns for all its features, derived from joining 7 different files (e.g., types, groups, dogmaAttributes, blueprints, etc.).

# Configuration

In [ ]:
import os
from google.colab import auth

# 1. Authenticate the user
print("Authenticating to Google Cloud...")
auth.authenticate_user()
print("Authentication complete.")

# 2. Set your project id
PROJECT_ID = "cloud-sa-ml"  # @param {type:"string"}
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

print(f"Set project to: {PROJECT_ID}")

# 3. Install the GCS library for pandas
print("Installing gcsfs...")
!pip install -q gcsfs
print("Installation complete.")

# 4. Set your Static data Bucket Name
BUCKET_NAME = "eve-online-foundation-data" # @param {type:"string"}
os.environ["EVE_DATA_BUCKET"] = BUCKET_NAME

BUCKET_PATH = f"gs://{BUCKET_NAME}"
print(f"Using GCS Bucket: {BUCKET_PATH}")


Authenticating to Google Cloud...
Authentication complete.
Set project to: cloud-sa-ml
Installing gcsfs...
Installation complete.
Using GCS Bucket: gs://eve-online-foundation-data


# Static Data Ingestion (Run onetime)

In [ ]:
%%bash

# --- 1. Script Setup ---
# Exit immediately if a command exits with a non-zero status.
set -e
# Treat unset variables as an error
set -u
# Ensure pipeline failures are captured
set -o pipefail

# --- 2. Configuration ---
ZIP_URL="https://developers.eveonline.com/static-data/eve-online-static-data-latest-jsonl.zip"
TEMP_DIR="eve_sde_data_temp"
ZIP_FILE="eve_sde.zip"

# --- 3. Cleanup Function ---
# This 'trap' will run the 'cleanup' function on script EXIT (whether success or failure)
function cleanup {
  echo "--- Running cleanup ---"
  rm -f "$ZIP_FILE"
  # Check if directory exists before trying to remove it
  if [ -d "$TEMP_DIR" ]; then
    rm -rf "$TEMP_DIR"
    echo "Removed $TEMP_DIR."
  fi
  echo "Cleanup complete."
}
trap cleanup EXIT

# --- 4. Main Execution ---

# Check for required tools
if ! command -v wget &> /dev/null || ! command -v unzip &> /dev/null || ! command -v gsutil &> /dev/null; then
  echo "Error: Required command (wget, unzip, or gsutil) not found."
  exit 1
fi

# Check if bucket exists and create if not
echo "Checking if bucket $BUCKET_PATH exists..."
if ! gsutil ls -b "$BUCKET_PATH" &> /dev/null; then
  echo "Bucket not found. Creating $BUCKET_PATH..."
  gsutil mb "$BUCKET_PATH"
else
  echo "Bucket $BUCKET_PATH already exists."
fi

# Step A: Download
echo "Downloading EVE SDE from $ZIP_URL..."
wget -O "$ZIP_FILE" "$ZIP_URL"
if [ $? -ne 0 ]; then
    echo "Download failed!"
    exit 1
fi
echo "Download complete."

# Step B: Unzip
echo "Unzipping files to $TEMP_DIR/..."
mkdir -p "$TEMP_DIR"
unzip -q "$ZIP_FILE" -d "$TEMP_DIR"
if [ $? -ne 0 ]; then
    echo "Unzip failed! The file might be corrupt."
    exit 1
fi
echo "Unzip complete."

# Step C: Find and Upload to GCS
echo "Finding .jsonl files..."
# Use 'find' to get a list of files and check if it's empty
# -L follows symlinks
file_list=$(find -L "$TEMP_DIR" -type f -name "*.jsonl")

if [ -z "$file_list" ]; then
  echo "Error: No .jsonl files were found in the unzipped archive."
  exit 1
fi

echo "Found .jsonl files. Uploading to gs://$BUCKET_NAME/..."
# Use the 'find' result to copy.
# We pipe the file list to gsutil -m cp -I for parallel streaming upload.
echo "$file_list" | gsutil -m cp -I "gs://$BUCKET_NAME/"
if [ $? -ne 0 ]; then
    echo "GCS upload failed!"
    exit 1
fi

# The 'trap' will handle cleanup automatically.
echo "--- All files successfully uploaded to gs://$BUCKET_NAME/ ---"
exit 0

Checking if bucket gs://eve-online-foundation-data/ exists...
Bucket gs://eve-online-foundation-data/ already exists.
Download complete.
Unzipping files to eve_sde_data_temp/...
Unzip complete.
Finding .jsonl files...
Found .jsonl files. Uploading to gs://eve-online-foundation-data/...
--- All files successfully uploaded to gs://eve-online-foundation-data/ ---
--- Running cleanup ---
Removed eve_sde_data_temp.
Cleanup complete.


--2025-10-24 21:29:24--  https://developers.eveonline.com/static-data/eve-online-static-data-latest-jsonl.zip
Resolving developers.eveonline.com (developers.eveonline.com)... 3.170.152.69, 3.170.152.87, 3.170.152.107, ...
Connecting to developers.eveonline.com (developers.eveonline.com)|3.170.152.69|:443... connected.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://developers.eveonline.com/static-data/tranquility/eve-online-static-data-3072925-jsonl.zip [following]
--2025-10-24 21:29:24--  https://developers.eveonline.com/static-data/tranquility/eve-online-static-data-3072925-jsonl.zip
Reusing existing connection to developers.eveonline.com:443.
HTTP request sent, awaiting response... 200 OK
Length: 82257287 (78M) [application/zip]
Saving to: ‘eve_sde.zip’

     0K .......... .......... .......... .......... ..........  0% 4.16M 19s
    50K .......... .......... .......... .......... ..........  0% 9.84M 13s
   100K .......... .......... .......... .....

# General EDA / Profiler

This cell is a generic "profiler" script used to explore the dataset.

It loops through a list of selected files (or all files), loads them, and automatically:

1. Flattens any dictionary columns (like name) to their English values.

2. Prints the df.info() to show schema, dtypes, and null counts.

3. Displays the df.head() to show sample data.

4. Prints value_counts() for any "safe" (hashable) categorical columns.

In [ ]:
import pandas as pd
import io
import sys
import time
from google.cloud import storage

# --- 1. Configuration ---
BUCKET_NAME = "eve-online-foundation-data"
GCS_PATH = f"gs://{BUCKET_NAME}"

# Set pandas options for nice output
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)

print(f"--- Starting General Profiler (with EN-Flattening) for gs://{BUCKET_NAME} ---")

# --- 2. Get Full List of Files ---
try:
    client = storage.Client()
    blobs = client.list_blobs(BUCKET_NAME)
    all_files = [blob.name for blob in blobs if blob.name.endswith('.jsonl')]
    print(f"Found {len(all_files)} total .jsonl files in the bucket.")
except Exception as e:
    print(f"Failed to list files: {e}")
    all_files = [] # empty list to avoid crash

# --- 3. SELECT WHICH FILES TO PROCESS ---

#files_to_process = [
#    'races.jsonl',
#    'factions.jsonl',
#    'categories.jsonl',
#    'groups.jsonl',
#    'mapRegions.jsonl', # This one will still show list data
#    'types.jsonl'
#]

#
# To run on ALL files, uncomment the next line
# WARNING: This will produce a *very* long output!
files_to_process = all_files
#

# Check if our selected files actually exist
files_to_process = [f for f in files_to_process if f in all_files]
print(f"Will process {len(files_to_process)} selected files: {files_to_process}\n")


# --- 4. Loop and Analyze ---
for filename in files_to_process:
    print(f"\n\n{'='*80}")
    print(f"--- 📊 Analyzing: {filename} ---")
    print(f"{'='*80}\n")

    try:
        # 1. Read into DF
        start_time = time.time()
        df = pd.read_json(f"{GCS_PATH}/{filename}", lines=True)
        print(f"Loaded {len(df)} rows in {time.time() - start_time:.2f}s")

        # --- 2. NEW: Pre-processing to flatten dicts ---
        print("\n[PRE-PROCESSING]")
        cols_to_flatten = []
        for col in df.columns:
            if df[col].dtype == 'object':
                # Find the first non-null item to check its type
                first_item = df[col].dropna().iloc[0] if not df[col].dropna().empty else None
                if isinstance(first_item, dict):
                    cols_to_flatten.append(col)

        if not cols_to_flatten:
            print("No dictionary columns found to flatten.")
        else:
            for col in cols_to_flatten:
                print(f"Flattening '{col}' to English ('en') key...")
                # Apply the logic: get 'en' value if it's a dict, otherwise keep original
                df[col] = df[col].apply(lambda x: x.get('en') if isinstance(x, dict) else x)

        # 3. Run df.info() (will show new flattened columns)
        print("\n[INFO]")
        info_buf = io.StringIO()
        df.info(buf=info_buf)
        print(info_buf.getvalue())

        # 4. Run df.head(5) (will show simple strings now)
        print("\n[HEAD]")
        display(df.head())

        # 5. Run value_counts()
        print("\n[VALUE COUNTS (Top 5 for hashable columns)]")

        if df.empty:
            print("DataFrame is empty.")
            continue

        for col in df.columns:
            if df[col].empty or df[col].isnull().all():
                print(f"\n--- Column: '{col}' (is empty or all null) ---")
                continue

            first_item = df[col].dropna().iloc[0]

            # Check for unhashable lists
            if isinstance(first_item, list):
                print(f"\n--- Column: '{col}' (Contains unhashable lists) ---")
                print("Sample value:", first_item)

            # All other types (str, int, float, bool) are now safe
            else:
                nunique = df[col].nunique()
                print(f"\n--- Column: '{col}' (Unique values: {nunique}) ---")

                # Show value_counts if it's categorical or low-cardinality
                if nunique < 100 or isinstance(first_item, str):
                    display(df[col].value_counts().head(5).to_frame())
                else:
                    print(f"Skipping value_counts (high-cardinality numeric: {nunique} unique)")

    except Exception as e:
        print(f"*** FAILED to process {filename}. Error: {e} ***")

print("\n--- Full Analysis Loop Finished ---")

--- Starting General Profiler (with EN-Flattening) for gs://eve-online-foundation-data ---
Found 52 total .jsonl files in the bucket.
Will process 52 selected files: ['_sde.jsonl', 'agentTypes.jsonl', 'agentsInSpace.jsonl', 'ancestries.jsonl', 'bloodlines.jsonl', 'blueprints.jsonl', 'categories.jsonl', 'certificates.jsonl', 'characterAttributes.jsonl', 'contrabandTypes.jsonl', 'controlTowerResources.jsonl', 'corporationActivities.jsonl', 'dbuffCollections.jsonl', 'dogmaAttributeCategories.jsonl', 'dogmaAttributes.jsonl', 'dogmaEffects.jsonl', 'dogmaUnits.jsonl', 'dynamicItemAttributes.jsonl', 'factions.jsonl', 'graphics.jsonl', 'groups.jsonl', 'icons.jsonl', 'landmarks.jsonl', 'mapAsteroidBelts.jsonl', 'mapConstellations.jsonl', 'mapMoons.jsonl', 'mapPlanets.jsonl', 'mapRegions.jsonl', 'mapSolarSystems.jsonl', 'mapStargates.jsonl', 'mapStars.jsonl', 'marketGroups.jsonl', 'masteries.jsonl', 'metaGroups.jsonl', 'npcCharacters.jsonl', 'npcCorporationDivisions.jsonl', 'npcCorporations.json

,_key,buildNumber,releaseDate
0,sde,3072925,2025-10-24T11:13:57Z



[VALUE COUNTS (Top 5 for hashable columns)]

--- Column: '_key' (Unique values: 1) ---


,count
_key,
sde,1



--- Column: 'buildNumber' (Unique values: 1) ---


...


--- Column: 'basePrice' (Unique values: 1535) ---
Skipping value_counts (high-cardinality numeric: 1535 unique)

--- Column: 'marketGroupID' (Unique values: 1559) ---
Skipping value_counts (high-cardinality numeric: 1559 unique)

--- Column: 'capacity' (Unique values: 329) ---
Skipping value_counts (high-cardinality numeric: 329 unique)

--- Column: 'metaGroupID' (Unique values: 13) ---


,count
metaGroupID,
4.0,2715
1.0,2615
2.0,2302
19.0,1433
17.0,1337



--- Column: 'variationParentTypeID' (Unique values: 1227) ---
Skipping value_counts (high-cardinality numeric: 1227 unique)

--- Column: 'factionID' (Unique values: 29) ---


,count
factionID,
500026.0,209
500027.0,165
500002.0,115
500004.0,112
500003.0,101



--- Full Analysis Loop Finished ---


The files fall under 3 main functional categories:

**1. Universe Geography (The "Map")**
This group defines the physical layout of the EVE universe. The files are all linked in a clear hierarchy (a region contains constellations, which contain solar systems, etc.).

* mapRegions.jsonl

* mapConstellations.jsonl

* mapSolarSystems.jsonl

* mapPlanets.jsonl

* mapMoons.jsonl

* mapStargates.jsonl

* mapStars.jsonl

**2. Item & Blueprint Definitions (The "Things")**
This group defines every single item in the game, from a ship to a piece of ore to a skillbook. It also uses a hierarchy (categories contain groups, which contain types).

* categories.jsonl (Highest level: "Ship", "Module", "Skill")

* groups.jsonl (Mid-level: "Frigate", "Cruiser", "Laser Turret")

* types.jsonl (The master file for all items: "Kestrel", "Rifter", "1MN Afterburner I")

* blueprints.jsonl (Defines manufacturing requirements)

* graphics.jsonl, icons.jsonl (Visual assets for the items)

**3. Factions, NPCs, and Stations (The "Entities")**
This group defines all the non-player characters and organizations, including who they are, who they work for, and where their stations are located.

* factions.jsonl (e.g., "Caldari State", "Gallente Federation")

* races.jsonl (e.g., "Caldari", "Gallente")

* npcCorporations.jsonl (The specific NPC companies)

* npcStations.jsonl (Links corporations to solar systems)

* agentsInSpace.jsonl (Links agents to corporations)



# Data Visualization (Plotly)

These cells use Plotly Express to create interactive visualizations for different aspects of the dataset. Each cell joins 2-4 files to build the necessary data.

* Treemap: Shows the hierarchy of all items (Categories -> Groups).

* Force-Directed Graph: Shows the ownership network (Factions -> Corporations -> Stations).

* Bar Chart: Shows the count of items by Race.

* 3D Scatter Plot: Creates a 3D map of the EVE universe by plotting mapSolarSystems.jsonl and coloring by mapRegions.jsonl.

## Treemap

In [ ]:
import pandas as pd
import plotly.express as px
import sys
import time
import io

# --- 1. Install Plotly ---
print("Installing plotly...")
!pip install -q plotly

# --- 2. Configuration ---
BUCKET_NAME = "eve-online-foundation-data"
GCS_PATH = f"gs://{BUCKET_NAME}"

# --- 3. Helper Function ---
def load_and_flatten_name(filename, gcs_path=GCS_PATH, id_col='_key', name_col='name'):
    """Loads a JSONL file from GCS, flattens its 'name' column, and renames its ID."""
    print(f"Loading & flattening: {filename}...")
    try:
        df = pd.read_json(f"{gcs_path}/{filename}", lines=True)

        # Flatten the name column (e.g., name['en'])
        if name_col in df.columns:
            new_col_name = f"{filename.split('.')[0]}_name_en"
            df[new_col_name] = df[name_col].apply(
                lambda x: x.get('en') if isinstance(x, dict) else None
            )
            # We don't drop the original 'name' in case we need it

        # Rename the primary key
        if id_col in df.columns:
            df = df.rename(columns={id_col: f'{filename.split(".")[0]}_id'})

        return df
    except Exception as e:
        print(f"FAILED to load {filename}. Error: {e}")
        return None

# --- 4. Main Data Loading and Merging ---
print("\n--- Building Hierarchy for Treemap ---")
start_time = time.time()

# Load the three key tables
df_types = load_and_flatten_name("types.jsonl")
df_groups = load_and_flatten_name("groups.jsonl")
df_categories = load_and_flatten_name("categories.jsonl")

# We only care about published items
df_types_published = df_types[df_types['published'] == True]

print("Merging tables...")
# Merge types -> groups
df_merged = pd.merge(
    df_types_published[['types_id', 'groupID', 'types_name_en']],
    df_groups[['groups_id', 'categoryID', 'groups_name_en']],
    left_on='groupID',
    right_on='groups_id',
    how='left'
)

# Merge result -> categories
df_full_hierarchy = pd.merge(
    df_merged,
    df_categories[['categories_id', 'categories_name_en']],
    left_on='categoryID',
    right_on='categories_id',
    how='left'
)

# Handle cases where items might not have a group or category
df_full_hierarchy['categories_name_en'] = df_full_hierarchy['categories_name_en'].fillna('Unknown Category')
df_full_hierarchy['groups_name_en'] = df_full_hierarchy['groups_name_en'].fillna('Unknown Group')

print(f"Data prepared in {time.time() - start_time:.2f}s")

# --- 5. Create the Treemap ---
print("\n--- Generating Treemap ---")
# To use the treemap, we need to group and count the items
df_grouped = df_full_hierarchy.groupby(
    ['categories_name_en', 'groups_name_en']
).size().reset_index(name='item_count')

# Create the plot
fig = px.treemap(
    df_grouped,
    # The 'path' defines the hierarchy. We add a root node "All Items".
    path=[px.Constant("All Published Items"), 'categories_name_en', 'groups_name_en'],
    # The 'values' defines the size of each box
    values='item_count',
    # 'color' can be used to shade by a value, here we use item_count again
    color='item_count',
    color_continuous_scale='Blues',
    title='EVE Online: Breakdown of All Published Items'
)

# Improve the hover data
fig.update_traces(
    hovertemplate='<b>%{label}</b><br>Item Count: %{value}<br>Parent: %{parent}'
)

fig.update_layout(margin = dict(t=50, l=25, r=25, b=25))

# Display the interactive plot in Colab
fig.show()

Installing plotly...

--- Building Hierarchy for Treemap ---
Loading & flattening: types.jsonl...
Loading & flattening: groups.jsonl...
Loading & flattening: categories.jsonl...
Merging tables...
Data prepared in 4.33s

--- Generating Treemap ---


## Faction & Station Ownership: Force-Directed Graph

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import networkx as nx # For graph layout
import sys
import time

# --- 1. Configuration (ensure GCS_PATH is defined) ---
BUCKET_NAME = "eve-online-foundation-data"
GCS_PATH = f"gs://{BUCKET_NAME}"

# --- 2. Helper Function (Copy from Cell 2) ---
def load_and_flatten_name(filename, gcs_path=GCS_PATH, id_col='_key', name_col='name'):
    """Loads a JSONL file from GCS, flattens its 'name' column, and renames its ID."""
    print(f"Loading & flattening: {filename}...")
    try:
        df = pd.read_json(f"{gcs_path}/{filename}", lines=True)

        # Flatten the name column (e.g., name['en'])
        if name_col in df.columns:
            new_col_name = f"{filename.split('.')[0]}_name_en"
            df[new_col_name] = df[name_col].apply(
                lambda x: x.get('en') if isinstance(x, dict) else None
            )
            # For network graphs, we often prefer a single name column
            df[name_col] = df[new_col_name] # Overwrite 'name' with English version
            df = df.drop(columns=[new_col_name]) # Drop the _name_en if 'name' is overwritten

        # Flatten 'description' column if present
        if 'description' in df.columns and any(isinstance(x, dict) for x in df['description'].dropna()):
            df['description'] = df['description'].apply(
                lambda x: x.get('en') if isinstance(x, dict) else x
            )

        # Rename the primary key
        if id_col in df.columns:
            df = df.rename(columns={id_col: f'{filename.split(".")[0]}_id'})

        return df
    except Exception as e:
        print(f"FAILED to load {filename}. Error: {e}")
        return None

# --- 3. Main Data Loading and Graph Construction (Corrected) ---
print("\n--- Building Data for Force-Directed Graph ---")
start_time = time.time()

# Load and flatten relevant dataframes
df_factions = load_and_flatten_name("factions.jsonl", name_col='name')
df_corporations = load_and_flatten_name("npcCorporations.jsonl", name_col='name')
# 'npcStations' does not have a 'name' column, so we don't pass 'name_col'
df_stations = load_and_flatten_name("npcStations.jsonl")
df_types = load_and_flatten_name("types.jsonl", name_col='name')

# --- Create Nodes ---
# Add a 'type' column for coloring/sizing
df_factions['node_type'] = 'Faction'
df_corporations['node_type'] = 'Corporation'
df_stations['node_type'] = 'Station'

# Get Station Type names
df_station_types = df_types[['types_id', 'name']].rename(columns={
    'types_id': 'typeID',
    'name': 'stationTypeName'
})
# Join the type name to the stations
df_stations = pd.merge(df_stations, df_station_types, on='typeID', how='left')

# Consolidate all nodes into a single DataFrame
all_nodes = pd.concat([
    # Factions
    df_factions.rename(columns={'factions_id': 'id', 'name': 'node_name'})
               [['id', 'node_name', 'node_type']],
    # Corporations
    df_corporations.rename(columns={'npcCorporations_id': 'id', 'name': 'node_name'})
                   [['id', 'node_name', 'node_type', 'factionID']],
    # Stations (using stationTypeName as the name)
    df_stations.rename(columns={'npcStations_id': 'id', 'stationTypeName': 'node_name', 'ownerID': 'corporationID'})
               [['id', 'node_name', 'node_type', 'corporationID']]
], ignore_index=True, sort=False)

# Clean up names (some station types might be null)
all_nodes['node_name'] = all_nodes['node_name'].fillna('Unknown Station Type')


# --- Create Edges (Links) ---
# Faction owns Corporation
faction_corp_edges = df_corporations.dropna(subset=['factionID']).apply(
    lambda row: {'source': row['factionID'], 'target': row['npcCorporations_id'], 'relation': 'Owns'}, axis=1
).tolist()

# Corporation owns Station
# Use 'ownerID' and 'npcStations_id' from the original df_stations
corp_station_edges = df_stations.dropna(subset=['ownerID']).apply(
    lambda row: {'source': row['ownerID'], 'target': row['npcStations_id'], 'relation': 'Owns'}, axis=1
).tolist()

all_edges = faction_corp_edges + corp_station_edges

# Filter out nodes not involved in any relationship to reduce clutter
involved_nodes_ids = set([edge['source'] for edge in all_edges] + [edge['target'] for edge in all_edges])
all_nodes_filtered = all_nodes[all_nodes['id'].isin(involved_nodes_ids)].copy()

print(f"Graph data prepared with {len(all_nodes_filtered)} nodes and {len(all_edges)} edges in {time.time() - start_time:.2f}s")

# --- 4. Build NetworkX Graph and Layout ---
G = nx.Graph()

for _, row in all_nodes_filtered.iterrows():
    G.add_node(row['id'],
               name=row['node_name'],
               type=row['node_type'],
               size=5 if row['node_type'] == 'Station' else (10 if row['node_type'] == 'Corporation' else 15), # Visual sizing
               hover_text=f"<b>{row['node_name']}</b><br>Type: {row['node_type']}<br>ID: {row['id']}")

for edge in all_edges:
    # Ensure source and target nodes exist in the filtered graph
    if edge['source'] in G and edge['target'] in G:
        G.add_edge(edge['source'], edge['target'], relation=edge['relation'])

# Use a spring layout for better visualization of clusters
pos = nx.spring_layout(G, k=0.1, iterations=50, seed=42) # k adjusts distance

# --- 5. Create Plotly Figure ---
print("--- Generating Force-Directed Graph ---")

edge_x = []
edge_y = []
for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.append(x0)
    edge_x.append(x1)
    edge_x.append(None) # Separates segments
    edge_y.append(y0)
    edge_y.append(y1)
    edge_y.append(None)

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=0.5, color='#888'),
    hoverinfo='none',
    mode='lines')

node_x = []
node_y = []
node_text = []
node_size = []
node_color = []

node_colors_map = {
    'Faction': '#FF5733',     # Red
    'Corporation': '#337BFF', # Blue
    'Station': '#33FF57'      # Green
}

for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)
    node_text.append(G.nodes[node]['hover_text'])
    node_size.append(G.nodes[node]['size'] * 2) # Adjust for better visibility
    node_color.append(node_colors_map.get(G.nodes[node]['type'], 'gray'))

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers',
    hoverinfo='text',
    text=node_text,
    marker=dict(
        showscale=False,
        color=node_color,
        size=node_size,
        line_width=2))

fig = go.Figure(data=[edge_trace, node_trace],
             layout=go.Layout(
                title='<br>EVE Online: Faction, Corporation, and Station Ownership Network',
                titlefont_size=16,
                showlegend=False,
                hovermode='closest',
                margin=dict(b=20,l=5,r=5,t=40),
                annotations=[ dict(
                    text="Red: Faction | Blue: Corporation | Green: Station",
                    showarrow=False,
                    xref="paper", yref="paper",
                    x=0.005, y=-0.002 ) ],
                xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
                )
fig.show()

print("Graph generation complete.")


--- Building Data for Force-Directed Graph ---
Loading & flattening: factions.jsonl...
Loading & flattening: npcCorporations.jsonl...
Loading & flattening: npcStations.jsonl...
Loading & flattening: types.jsonl...
Graph data prepared with 5455 nodes and 5427 edges in 7.34s
--- Generating Force-Directed Graph ---


Graph generation complete.


## Bar Chart (Items by Race)

In [ ]:
import pandas as pd
import plotly.express as px
import sys
import time
import io

# --- 1. Configuration (ensure GCS_PATH and helper function are in memory) ---
# This assumes GCS_PATH is set and load_and_flatten_name() is defined
# in a previous cell (like Cell 2).

print("--- Building Bar Chart: Item Types by Race ---")
start_time = time.time()

try:
    # --- 2. Load Data ---
    print("Loading types.jsonl (only key columns)...")
    # We can optimize by only loading the columns we need
    df_types = pd.read_json(
        f"{GCS_PATH}/types.jsonl",
        lines=True,
        dtype={'raceID': 'float'} # Ensure raceID is float to handle NaNs
    )
    # Filter for published items and keep only what we need
    df_types_published = df_types[
        df_types['published'] == True
    ][['_key', 'raceID']].rename(columns={'_key': 'types_id'})

    # Load and flatten races
    df_races = load_and_flatten_name("races.jsonl", name_col='name')
    df_races = df_races[['races_id', 'name']] # Keep only ID and name

    print("Merging data...")
    # --- 3. Merge Data ---
    df_merged = pd.merge(
        df_types_published,
        df_races,
        left_on='raceID',
        right_on='races_id',
        how='left' # Keep all types, even those with no race
    )

    # --- 4. Clean and Group ---
    # Items with no race (raceID is NaN) are important! Let's label them.
    df_merged['name'] = df_merged['name'].fillna('Race: None')

    print("Grouping and counting...")
    df_counts = df_merged.groupby('name').size().reset_index(name='item_count')
    df_counts = df_counts.sort_values(by='item_count', ascending=False)

    print(f"Data prepared in {time.time() - start_time:.2f}s")

    # --- 5. Plot ---
    print("\n--- Generating Bar Chart ---")
    fig = px.bar(
        df_counts,
        x='name',
        y='item_count',
        color='name',
        title='EVE Online: Count of Published Items by Race',
        labels={'name': 'Race', 'item_count': 'Number of Published Items'}
    )

    fig.update_layout(xaxis_title="Race", yaxis_title="Item Count")
    fig.show()

except Exception as e:
    print(f"*** FAILED to build bar chart. Error: {e} ***")

--- Building Bar Chart: Item Types by Race ---
Loading types.jsonl (only key columns)...
Loading & flattening: races.jsonl...
Merging data...
Grouping and counting...
Data prepared in 7.96s

--- Generating Bar Chart ---


## Spatial Data: 3D Scatter Plots

In [ ]:
import pandas as pd
import plotly.express as px
import sys
import time
import io

# --- 1. Configuration (ensure GCS_PATH and helper function are in memory) ---
# This assumes GCS_PATH is set and load_and_flatten_name() is defined
# in a previous cell (like Cell 2).

print("--- Building 3D Solar System Map ---")
start_time = time.time()

try:
    # --- 2. Load Data ---
    print("Loading mapSolarSystems.jsonl...")
    df_systems = load_and_flatten_name("mapSolarSystems.jsonl", name_col='name')
    # Rename the flattened name for clarity
    df_systems = df_systems.rename(columns={'name': 'system_name_en'})

    print("Loading mapRegions.jsonl...")
    df_regions = load_and_flatten_name("mapRegions.jsonl", name_col='name')

    # Use the correct 'mapRegions_id'
    df_regions = df_regions[['mapRegions_id', 'name']].rename(columns={'name': 'region_name_en'})

    print("Merging data...")
    # --- 3. Merge Data ---
    df_merged = pd.merge(
        df_systems,
        df_regions,
        left_on='regionID',
        right_on='mapRegions_id', # Corrected column name
        how='left'
    )

    # Handle any systems without a region
    df_merged['region_name_en'] = df_merged['region_name_en'].fillna('Unknown Region')

    # --- 4. Un-nest the 'position' column ---
    print("Un-nesting 'position' column...")
    pos_df = pd.json_normalize(df_merged['position'])

    # Join the new x, y, z columns back to the main DataFrame
    df_final_map = pd.concat([df_merged, pos_df], axis=1)

    print(f"Data prepared in {time.time() - start_time:.2f}s")

    # --- 5. Plot ---
    print("\n--- Generating 3D Scatter Plot ---")

    fig = px.scatter_3d(
        df_final_map, # Use the final, un-nested DataFrame
        x='x',
        y='y',
        z='z',
        color='region_name_en',      # Color-code by region name
        hover_name='system_name_en', # Show system name on hover
        hover_data={                 # Add extra data to hover
            'region_name_en': True,
            'securityStatus': True,
            'x': False, # Hide coordinates from hover box
            'y': False,
            'z': False
        },
        title='3D Map of the EVE Online Universe'
    )

    # --- 6. Clean up the plot ---
    fig.update_traces(marker=dict(size=1.5))
    fig.update_layout(
        title_text='3D Map of the EVE Online Universe (Colored by Region)',
        scene=dict(
            xaxis_title='X Coordinate',
            yaxis_title='Y Coordinate',
            zaxis_title='Z Coordinate',
            bgcolor='rgb(10, 10, 10)' # Dark background
        ),
        legend_title_text='Region'
    )

    fig.show()

except Exception as e:
    print(f"*** FAILED to build 3D map. Error: {e} ***")

--- Building 3D Solar System Map ---
Loading mapSolarSystems.jsonl...
Loading & flattening: mapSolarSystems.jsonl...
Loading mapRegions.jsonl...
Loading & flattening: mapRegions.jsonl...
Merging data...
Un-nesting 'position' column...
Data prepared in 0.59s

--- Generating 3D Scatter Plot ---


# Helper Functions

In [ ]:
import pandas as pd
import sys

# --- 1. Configuration (ensure GCS_PATH is defined) ---
BUCKET_NAME = "eve-online-foundation-data"
GCS_PATH = f"gs://{BUCKET_NAME}"

# --- 2. Helper Function (Copy from Cell 2) ---
def load_and_flatten_name(filename, gcs_path=GCS_PATH, id_col='_key', name_col='name'):
    """Loads a JSONL file from GCS, flattens its 'name' column, and renames its ID."""
    print(f"Loading & flattening: {filename}...")
    try:
        df = pd.read_json(f"{gcs_path}/{filename}", lines=True)

        # Flatten the name column (e.g., name['en'])
        if name_col in df.columns:
            new_col_name = f"{filename.split('.')[0]}_name_en"
            df[new_col_name] = df[name_col].apply(
                lambda x: x.get('en') if isinstance(x, dict) else None
            )
            # For network graphs, we often prefer a single name column
            df[name_col] = df[new_col_name] # Overwrite 'name' with English version
            df = df.drop(columns=[new_col_name]) # Drop the _name_en if 'name' is overwritten

        # Flatten 'description' column if present
        if 'description' in df.columns and any(isinstance(x, dict) for x in df['description'].dropna()):
            df['description'] = df['description'].apply(
                lambda x: x.get('en') if isinstance(x, dict) else x
            )

        # Rename the primary key
        if id_col in df.columns:
            df = df.rename(columns={id_col: f'{filename.split(".")[0]}_id'})

        return df
    except Exception as e:
        print(f"FAILED to load {filename}. Error: {e}")
        return None

# --- 3. Run the Debug ---
print("--- Debugging DataFrame Schemas ---")

try:
    print("\n[INFO] df_factions:")
    df_factions = load_and_flatten_name("factions.jsonl", name_col='name')
    df_factions.info()

    print("\n[INFO] df_corporations:")
    df_corporations = load_and_flatten_name("npcCorporations.jsonl", name_col='name')
    df_corporations.info()

    print("\n[INFO] df_stations:")
    df_stations = load_and_flatten_name("npcStations.jsonl", name_col='name')
    df_stations.info()

except Exception as e:
    print(f"\n*** FAILED during debug. Error: {e} ***")

print("\n--- Debug Script Finished ---")

--- Debugging DataFrame Schemas ---

[INFO] df_factions:
Loading & flattening: factions.jsonl...
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   factions_id           27 non-null     int64  
 1   corporationID         26 non-null     float64
 2   description           27 non-null     object 
 3   flatLogo              18 non-null     object 
 4   flatLogoWithName      6 non-null      object 
 5   iconID                27 non-null     int64  
 6   memberRaces           27 non-null     object 
 7   militiaCorporationID  6 non-null      float64
 8   name                  27 non-null     object 
 9   shortDescription      4 non-null      object 
 10  sizeFactor            27 non-null     int64  
 11  solarSystemID         27 non-null     int64  
 12  uniqueName            27 non-null     bool   
dtypes: bool(1), float64(2), int64(

In [ ]:
import pandas as pd
import sys
import time
import io

# --- Configuration (Global) ---
BUCKET_NAME = "eve-online-foundation-data"
GCS_PATH = f"gs://{BUCKET_NAME}"

def load_and_flatten_name(filename, gcs_path=GCS_PATH, id_col='_key', name_col='name'):
    """Loads a JSONL file from GCS, flattens its 'name' column, and renames its ID."""
    print(f"Loading & flattening: {filename}...")
    try:
        df = pd.read_json(f"{gcs_path}/{filename}", lines=True)

        # Flatten the name column (e.g., name['en'])
        if name_col in df.columns:
            df[f'{filename.split(".")[0]}_name_en'] = df[name_col].apply(
                lambda x: x.get('en') if isinstance(x, dict) else None
            )
            df = df.drop(columns=[name_col])

        # Rename the primary key
        if id_col in df.columns:
            df = df.rename(columns={id_col: f'{filename.split(".")[0]}_id'})

        return df
    except Exception as e:
        print(f"FAILED to load {filename}. Error: {e}")
        return None

def build_dogma_features(gcs_path, attribute_names_to_get):
    """Builds a pivoted feature table for item stats (Dogma)."""
    print("Building Dogma (stats) feature table...")
    try:
        # 1. Load attribute names
        print("Loading: dogmaAttributes.jsonl...")
        df_attr = pd.read_json(f"{gcs_path}/dogmaAttributes.jsonl", lines=True)
        df_attr = df_attr.rename(columns={'_key': 'attributeID'})
        df_attr = df_attr[['attributeID', 'name']] # Use the raw 'name' column

        # 2. Filter for ONLY the attributes we care about
        df_attr_filtered = df_attr[df_attr['name'].isin(attribute_names_to_get)]
        print(f"Filtered to {len(df_attr_filtered)} key attributes.")

        if len(df_attr_filtered) == 0:
            print("*** WARNING: Filtered to 0 attributes. Check your attribute list.")
            return pd.DataFrame(columns=['typeID']) # Return empty DF

        # 3. Load and un-nest the typeDogma data
        print("Loading and un-nesting typeDogma.jsonl...")
        df_dogma_nested = pd.read_json(f"{gcs_path}/typeDogma.jsonl", lines=True)
        df_dogma_nested = df_dogma_nested.rename(columns={'_key': 'typeID'})

        df_dogma_long = df_dogma_nested.explode('dogmaAttributes')
        df_dogma_long = df_dogma_long.dropna(subset=['dogmaAttributes'])

        df_dogma_attrs = pd.json_normalize(df_dogma_long['dogmaAttributes'])
        df_dogma_long = df_dogma_long.reset_index(drop=True)

        df_dogma_flat = pd.concat([df_dogma_long[['typeID']], df_dogma_attrs], axis=1)
        print("Successfully un-nested typeDogma data.")

        # 6. Merge dogma values with our filtered attribute names
        df_merged_dogma = pd.merge(
            df_dogma_flat,
            df_attr_filtered[['attributeID', 'name']],
            on='attributeID',
            how='inner'
        )

        # 7. Pivot the table!
        print("Pivoting Dogma table...")
        df_dogma_features = df_merged_dogma.pivot_table(
            index='typeID',
            columns='name',
            values='value',
            aggfunc='first'
        )

        df_dogma_features = df_dogma_features.add_prefix('dogma_')
        print("Dogma feature table built successfully.")
        return df_dogma_features.reset_index() # Make 'typeID' a column

    except Exception as e:
        print(f"FAILED to build dogma features. Error: {e}")
        return None

print("All helper functions defined.")

All helper functions defined.


# Feature Vectors

## Feature Engineering: Ship Stats Vector
Build Ship + Stats Vector

In [ ]:
# Set pandas options for nice output
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)

def main_build_ship_vector():
    start_time = time.time()

    # 1. Define the features we want
    # --- FIX 1: 'basePrice' is REMOVED from this list ---
    SHIP_STATS_TO_GET = [
        'cpuOutput', 'powergridOutput', 'capacitorCapacity',
        'shieldCapacity', 'shieldEmDamageResonance', 'shieldThermalDamageResonance',
        'shieldKineticDamageResonance', 'shieldExplosiveDamageResonance',
        'armorHP', 'armorEmDamageResonance', 'armorThermalDamageResonance',
        'armorKineticDamageResonance', 'armorExplosiveDamageResonance',
        'structureHP', 'mass', 'volume', 'capacity'
    ]

    # 2. Build the Dogma (Stats) Feature Table
    dogma_features = build_dogma_features(GCS_PATH, SHIP_STATS_TO_GET)
    if dogma_features is None:
        sys.exit("Stopping due to dogma build failure.")

    # 3. Load dimension tables
    # 'basePrice' is loaded here from types.jsonl
    df_types = load_and_flatten_name("types.jsonl", GCS_PATH, "_key", "name")
    df_groups = load_and_flatten_name("groups.jsonl", GCS_PATH, "_key", "name")
    df_categories = load_and_flatten_name("categories.jsonl", GCS_PATH, "_key", "name")
    df_market_groups = load_and_flatten_name("marketGroups.jsonl", GCS_PATH, "_key", "name")

    # 4. Build the "Base Ship" DataFrame
    print("Building base ship table...")
    df_base = pd.merge(df_types, df_groups, left_on='groupID', right_on='groups_id', how='left')
    df_base = pd.merge(df_base, df_categories, left_on='categoryID', right_on='categories_id', how='left')

    df_ships = df_base[
        (df_base['categories_name_en'] == 'Ship') &
        (df_base['published'] == True)
    ].copy()
    print(f"Found {len(df_ships)} published ship types.")

    # 5. Join all feature tables
    print("Joining all feature tables...")
    df_final_vector = pd.merge(
        df_ships,
        df_market_groups[['marketGroups_id', 'marketGroups_name_en']],
        left_on='marketGroupID',
        right_on='marketGroups_id',
        how='left'
    )

    if 'typeID' in dogma_features.columns and len(dogma_features) > 0:
        df_final_vector = pd.merge(
            df_final_vector,
            dogma_features,
            left_on='types_id',
            right_on='typeID',
            how='left'
        )

    # 6. Clean and Report
    stat_cols = [col for col in df_final_vector.columns if col.startswith('dogma_')]
    df_final_vector[stat_cols] = df_final_vector[stat_cols].fillna(0)

    # --- FIX 2: 'basePrice' is ADDED to this list ---
    final_columns_to_keep = [
        'types_id', 'types_name_en', 'groups_name_en', 'categories_name_en',
        'marketGroups_name_en', 'basePrice' # <--- ADDED HERE
    ] + stat_cols

    final_columns_to_keep = [col for col in final_columns_to_keep if col in df_final_vector.columns]

    df_final_vector = df_final_vector[final_columns_to_keep].rename(columns={
        'types_id': 'ship_typeID',
        'types_name_en': 'ship_name',
        'groups_name_en': 'group_name',
        'categories_name_en': 'category_name',
        'marketGroups_name_en': 'market_group'
    })

    print(f"\n--- Feature Vector Build Complete! (Total time: {time.time() - start_time:.2f}s) ---")

    print("\n[INFO] Final Ship Feature Vector:")
    df_final_vector.info()

    print("\n[HEAD] Final Ship Feature Vector:")
    display(df_final_vector.head()) # 'display()' gives a nice HTML table in Colab

    return df_final_vector # Return the DataFrame for later use

# --- Run the analysis ---
ship_stats_vector = main_build_ship_vector()

Building Dogma (stats) feature table...
Loading: dogmaAttributes.jsonl...
Filtered to 15 key attributes.
Loading and un-nesting typeDogma.jsonl...
Successfully un-nested typeDogma data.
Pivoting Dogma table...
Dogma feature table built successfully.
Loading & flattening: types.jsonl...
Loading & flattening: groups.jsonl...
Loading & flattening: categories.jsonl...
Loading & flattening: marketGroups.jsonl...
Building base ship table...
Found 550 published ship types.
Joining all feature tables...

--- Feature Vector Build Complete! (Total time: 8.26s) ---

[INFO] Final Ship Feature Vector:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 550 entries, 0 to 549
Data columns (total 20 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   ship_typeID                           550 non-null    int64  
 1   ship_name                             550 non-null    object 
 2   group_name                

,ship_typeID,ship_name,group_name,category_name,market_group,basePrice,dogma_armorEmDamageResonance,dogma_armorExplosiveDamageResonance,dogma_armorHP,dogma_armorKineticDamageResonance,dogma_armorThermalDamageResonance,dogma_capacitorCapacity,dogma_cpuOutput,dogma_mass,dogma_shieldCapacity,dogma_shieldEmDamageResonance,dogma_shieldExplosiveDamageResonance,dogma_shieldKineticDamageResonance,dogma_shieldThermalDamageResonance,dogma_volume
0,582,Bantam,Frigate,Ship,Caldari,300000.0,0.5,0.9,225.0,0.75,0.55,615.0,200.0,0.0,500.0,1.0,0.5,0.6,0.8,0.0
1,583,Condor,Frigate,Ship,Caldari,400000.0,0.5,0.9,250.0,0.75,0.55,300.0,185.0,0.0,400.0,1.0,0.5,0.6,0.8,0.0
2,584,Griffin,Frigate,Ship,Caldari,300000.0,0.5,0.9,250.0,0.75,0.55,245.0,240.0,0.0,400.0,1.0,0.5,0.6,0.8,0.0
3,585,Slasher,Frigate,Ship,Minmatar,400000.0,0.4,0.9,300.0,0.75,0.65,240.0,140.0,0.0,350.0,1.0,0.5,0.6,0.8,0.0
4,586,Probe,Frigate,Ship,Minmatar,400000.0,0.4,0.9,300.0,0.75,0.65,235.0,240.0,0.0,300.0,1.0,0.5,0.6,0.8,0.0


## Feature Engineering: Manufacturing Vector

### Manufacturing Files

In [ ]:
print("--- 1. Understanding 'blueprints.jsonl' ---")
try:
    # Load just the first 5 rows to see the structure
    df_blueprints = pd.read_json(f"{GCS_PATH}/blueprints.jsonl", lines=True, nrows=5)

    print("\n[INFO] 'blueprints.jsonl':")
    df_blueprints.info()

    print("\n[HEAD] 'blueprints.jsonl':")
    display(df_blueprints.head())

except Exception as e:
    print(f"FAILED to load blueprints.jsonl. Error: {e}")

print("\n--- 2. Understanding 'typeMaterials.jsonl' ---")
try:
    # Load just the first 5 rows
    df_materials = pd.read_json(f"{GCS_PATH}/typeMaterials.jsonl", lines=True, nrows=5)

    print("\n[INFO] 'typeMaterials.jsonl':")
    df_materials.info()

    print("\n[HEAD] 'typeMaterials.jsonl':")
    display(df_materials.head())

except Exception as e:
    print(f"FAILED to load typeMaterials.jsonl. Error: {e}")

print("\n--- Debugging Script Finished ---")

--- 1. Understanding 'blueprints.jsonl' ---

[INFO] 'blueprints.jsonl':
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   _key                5 non-null      int64 
 1   activities          5 non-null      object
 2   blueprintTypeID     5 non-null      int64 
 3   maxProductionLimit  5 non-null      int64 
dtypes: int64(3), object(1)
memory usage: 292.0+ bytes

[HEAD] 'blueprints.jsonl':


,_key,activities,blueprintTypeID,maxProductionLimit
0,681,"{'copying': {'time': 480}, 'manufacturing': {'...",681,300
1,682,"{'copying': {'time': 480}, 'manufacturing': {'...",682,300
2,683,"{'copying': {'time': 4800}, 'invention': {'mat...",683,30
3,684,"{'copying': {'time': 4800}, 'invention': {'mat...",684,30
4,685,"{'copying': {'time': 4800}, 'invention': {'mat...",685,30



--- 2. Understanding 'typeMaterials.jsonl' ---

[INFO] 'typeMaterials.jsonl':
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   _key       5 non-null      int64 
 1   materials  5 non-null      object
dtypes: int64(1), object(1)
memory usage: 212.0+ bytes

[HEAD] 'typeMaterials.jsonl':


,_key,materials
0,18,"[{'materialTypeID': 34, 'quantity': 175}, {'ma..."
1,19,"[{'materialTypeID': 34, 'quantity': 48000}, {'..."
2,20,"[{'materialTypeID': 36, 'quantity': 60}, {'mat..."
3,21,"[{'materialTypeID': 35, 'quantity': 450}, {'ma..."
4,22,"[{'materialTypeID': 35, 'quantity': 3200}, {'m..."



--- Debugging Script Finished ---


from matplotlib import pyplot as plt
_df_0['_key'].plot(kind='hist', bins=20, title='_key')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  counted = (series['_key']
                .value_counts()
              .reset_index(name='counts')
              .rename({'index': '_key'}, axis=1)
              .sort_values('_key', ascending=True))
  xs = counted['_key']
  ys = counted['counts']
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_1.sort_values('_key', ascending=True)
_plot_series(df_sorted, '')
sns.despine(fig=fig, ax=ax)
plt.xlabel('_key')
_ = plt.ylabel('count()')

from matplotlib import pyplot as plt
_df_2['_key'].plot(kind='line', figsize=(8, 4), title='_key')
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
import pandas as pd
import json

GCS_PATH = f"gs://{BUCKET_NAME}"

print("--- Understanding 'blueprints.jsonl' activities column ---")
try:
    # Load just the Bantam Blueprint (typeID 681)
    df_blueprints = pd.read_json(f"{GCS_PATH}/blueprints.jsonl", lines=True)

    # Let's find the 'Bantam Blueprint' (blueprintTypeID 681)
    bantam_bp = df_blueprints[df_blueprints['blueprintTypeID'] == 681].iloc[0]

    print("\n[INFO] Full 'activities' data for 'Bantam Blueprint':\n")

    # Pretty-print the JSON from the 'activities' column
    print(json.dumps(bantam_bp['activities'], indent=2))

except Exception as e:
    print(f"FAILED to load and inspect blueprint. Error: {e}")

print("\n--- Debug Script Finished ---")

--- Understanding 'blueprints.jsonl' activities column ---

[INFO] Full 'activities' data for 'Bantam Blueprint':

{
  "copying": {
    "time": 480
  },
  "manufacturing": {
    "materials": [
      {
        "quantity": 86,
        "typeID": 38
      }
    ],
    "products": [
      {
        "quantity": 1,
        "typeID": 165
      }
    ],
    "time": 600
  },
  "research_material": {
    "time": 210
  },
  "research_time": {
    "time": 210
  }
}

--- Debug Script Finished ---


## Build Manufacturing Anomaly Report

In [ ]:
import pandas as pd
import sys
import time
import io

# Set pandas options for nice output
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)

def main_build_mfg_vector():
    start_time = time.time()

    # --- 1. Load Base Ship & Type Data ---
    # We need df_types for material names later
    print("Loading base types, groups, and categories...")
    df_types = load_and_flatten_name("types.jsonl", GCS_PATH, "_key", "name")
    df_groups = load_and_flatten_name("groups.jsonl", GCS_PATH, "_key", "name")
    df_categories = load_and_flatten_name("categories.jsonl", GCS_PATH, "_key", "name")

    df_base = pd.merge(df_types, df_groups, left_on='groupID', right_on='groups_id', how='left')
    df_base = pd.merge(df_base, df_categories, left_on='categoryID', right_on='categories_id', how='left')

    df_ships = df_base[
        (df_base['categories_name_en'] == 'Ship') &
        (df_base['published'] == True)
    ][['types_id', 'types_name_en', 'groups_name_en']].rename(columns={
        'types_id': 'ship_typeID',
        'types_name_en': 'ship_name',
        'groups_name_en': 'ship_group'
    })
    print(f"Found {len(df_ships)} published ships.")

    # --- 2. Load and UN-NEST Blueprint Data ---
    print("Loading and un-nesting 'blueprints.jsonl'...")
    try:
        df_blueprints = pd.read_json(f"{GCS_PATH}/blueprints.jsonl", lines=True)
        df_blueprints = df_blueprints.rename(columns={'_key': 'blueprint_typeID'})

        # Normalize the 'activities' column to get 'manufacturing.products' and 'manufacturing.materials'
        df_activities = pd.json_normalize(df_blueprints['activities'])

        # Combine with the blueprint's ID
        df_blueprints_flat = pd.concat([df_blueprints[['blueprint_typeID']], df_activities], axis=1)

        # --- 2a. Process Products ---
        print("...processing manufacturing products.")
        # We must check if 'manufacturing.products' exists
        if 'manufacturing.products' not in df_blueprints_flat.columns:
            raise KeyError("Column 'manufacturing.products' not found.")

        df_bp_prod = df_blueprints_flat[['blueprint_typeID', 'manufacturing.products']].dropna(subset=['manufacturing.products'])
        df_bp_prod_exploded = df_bp_prod.explode('manufacturing.products')
        df_bp_prod_norm = pd.json_normalize(df_bp_prod_exploded['manufacturing.products'])

        # Our final product lookup table: [blueprint_typeID, productTypeID]
        df_bp_products_lookup = pd.concat([
            df_bp_prod_exploded[['blueprint_typeID']].reset_index(drop=True),
            df_bp_prod_norm[['typeID']]
        ], axis=1).rename(columns={'typeID': 'productTypeID'})

        # --- 2b. Process Materials ---
        print("...processing manufacturing materials.")
        if 'manufacturing.materials' not in df_blueprints_flat.columns:
            raise KeyError("Column 'manufacturing.materials' not found.")

        df_bp_mat = df_blueprints_flat[['blueprint_typeID', 'manufacturing.materials']].dropna(subset=['manufacturing.materials'])
        df_bp_mat_exploded = df_bp_mat.explode('manufacturing.materials')
        df_bp_mat_norm = pd.json_normalize(df_bp_mat_exploded['manufacturing.materials'])

        # Our final material lookup table: [blueprint_typeID, materialTypeID, quantity]
        df_bp_materials_long = pd.concat([
            df_bp_mat_exploded[['blueprint_typeID']].reset_index(drop=True),
            df_bp_mat_norm[['typeID', 'quantity']]
        ], axis=1).rename(columns={'typeID': 'materialTypeID'})

        print("Successfully un-nested blueprints.")

    except Exception as e:
        print(f"*** FAILED to un-nest 'blueprints.jsonl'. Error: {e}")
        return

    # --- 3. Pivot Material Costs ---
    print("Pivoting material costs...")

    # We need the *names* of the materials (e.g., "Tritanium") from the types table
    df_material_names = df_types[['types_id', 'types_name_en']].rename(columns={
        'types_id': 'materialTypeID',
        'types_name_en': 'material_name'
    })

    df_materials_named = pd.merge(df_bp_materials_long, df_material_names, on='materialTypeID', how='left')

    # Pivot the table to get columns like 'mfg_Tritanium', 'mfg_Pyerite'
    KEY_MINERALS = ['Tritanium', 'Pyerite', 'Mexallon', 'Isogen', 'Nocxium', 'Zydrine', 'Megacyte']
    df_materials_filtered = df_materials_named[df_materials_named['material_name'].isin(KEY_MINERALS)]

    df_mfg_features = df_materials_filtered.pivot_table(
        index='blueprint_typeID', # Use blueprint_typeID as the key
        columns='material_name',
        values='quantity',
        aggfunc='sum'
    ).add_prefix('mfg_')

    print("Manufacturing feature table built.")

    # --- 4. Join Features to Ship List ---
    print("Joining features to ship list...")

    # 1. Which ships have a blueprint? (Join ships with blueprint products)
    df_final = pd.merge(
        df_ships,
        df_bp_products_lookup,
        left_on='ship_typeID',
        right_on='productTypeID',
        how='left' # Keep all ships, even those with no blueprint
    )

    # 2. What are the costs for that blueprint?
    # (Join blueprint_typeID with the material cost table)
    df_final = pd.merge(
        df_final,
        df_mfg_features,
        on='blueprint_typeID', # This is now the common key
        how='left'
    )

    # --- 5. Clean and Report ---
    mfg_cols = [col for col in df_final.columns if col.startswith('mfg_')]
    df_final[mfg_cols] = df_final[mfg_cols].fillna(0) # Ships that don't need Tritanium

    # A 'NaN' blueprint_typeID means the ship cannot be built.
    df_final['is_buildable'] = df_final['blueprint_typeID'].notna()

    print(f"\n--- Augmentation Complete! (Total time: {time.time() - start_time:.2f}s) ---")

    print("\n[HEAD] Ships and their mineral costs:")
    display(df_final[df_final['is_buildable'] == True][['ship_name'] + mfg_cols].head())

    print("\n--- ANOMALY REPORT: Published Ships That Are NOT Buildable ---")
    anomaly_df = df_final[df_final['is_buildable'] == False][['ship_name', 'ship_group', 'is_buildable']]
    display(anomaly_df.head(20))

    return df_final, anomaly_df

# --- Run the analysis ---
mfg_vector, anomalies = main_build_mfg_vector()

Loading base types, groups, and categories...
Loading & flattening: types.jsonl...
Loading & flattening: groups.jsonl...
Loading & flattening: categories.jsonl...
Found 550 published ships.
Loading and un-nesting 'blueprints.jsonl'...
...processing manufacturing products.
...processing manufacturing materials.
Successfully un-nested blueprints.
Pivoting material costs...
Manufacturing feature table built.
Joining features to ship list...

--- Augmentation Complete! (Total time: 5.22s) ---

[HEAD] Ships and their mineral costs:


,ship_name,mfg_Isogen,mfg_Megacyte,mfg_Mexallon,mfg_Nocxium,mfg_Pyerite,mfg_Tritanium,mfg_Zydrine
0,Bantam,375.0,0.0,1875.0,0.0,4500.0,24000.0,0.0
1,Condor,500.0,0.0,2500.0,0.0,6000.0,32000.0,0.0
2,Griffin,375.0,0.0,1875.0,0.0,4500.0,24000.0,0.0
3,Slasher,500.0,0.0,2500.0,0.0,6000.0,32000.0,0.0
4,Probe,550.0,0.0,2750.0,0.0,6600.0,35200.0,0.0



--- ANOMALY REPORT: Published Ships That Are NOT Buildable ---


,ship_name,ship_group,is_buildable
6,Reaper,Corvette,False
13,Gallente Police Ship,Frigate,False
14,Impairor,Corvette,False
19,Ibis,Corvette,False
23,Velator,Corvette,False
49,Opux Luxury Yacht,Cruiser,False
68,Capsule,Capsule,False
71,Polaris Enigma Frigate,Corvette,False
72,Concord Police Frigate,Frigate,False
73,Concord SWAT Frigate,Frigate,False


## Create Master Feature Vector

In [ ]:
import pandas as pd

# This cell assumes 'ship_stats_vector' (from Cell 3)
# and 'mfg_vector' (from Cell 4) are in memory.

print("--- 1. Creating Master Feature Vector ---")

# We only need the manufacturing columns from the mfg_vector
mfg_cols_to_merge = [
    'ship_typeID',
    'is_buildable',
    'blueprint_typeID'
] + [col for col in mfg_vector.columns if col.startswith('mfg_')]

# Merge the stats vector (Cell 3) with the mfg vector (Cell 4)
master_vector = pd.merge(
    ship_stats_vector,
    mfg_vector[mfg_cols_to_merge],
    on='ship_typeID',
    how='left'
)

# Fill NaNs created by the merge (e.g., non-buildable ships)
mfg_cols = [col for col in master_vector.columns if col.startswith('mfg_')]
master_vector[mfg_cols] = master_vector[mfg_cols].fillna(0)
master_vector['is_buildable'] = master_vector['is_buildable'].fillna(False)

print("Master vector created successfully.")

print("\n[INFO] Master Ship Feature Vector:")
master_vector.info()

print("\n[HEAD] Master Ship Feature Vector:")
display(master_vector.head())

--- 1. Creating Master Feature Vector ---
Master vector created successfully.

[INFO] Master Ship Feature Vector:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 550 entries, 0 to 549
Data columns (total 29 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   ship_typeID                           550 non-null    int64  
 1   ship_name                             550 non-null    object 
 2   group_name                            550 non-null    object 
 3   category_name                         550 non-null    object 
 4   market_group                          407 non-null    object 
 5   basePrice                             465 non-null    float64
 6   dogma_armorEmDamageResonance          550 non-null    float64
 7   dogma_armorExplosiveDamageResonance   550 non-null    float64
 8   dogma_armorHP                         550 non-null    float64
 9   dogma_armorKineticDamageResonance     5

,ship_typeID,ship_name,group_name,category_name,market_group,basePrice,dogma_armorEmDamageResonance,dogma_armorExplosiveDamageResonance,dogma_armorHP,dogma_armorKineticDamageResonance,dogma_armorThermalDamageResonance,dogma_capacitorCapacity,dogma_cpuOutput,dogma_mass,dogma_shieldCapacity,dogma_shieldEmDamageResonance,dogma_shieldExplosiveDamageResonance,dogma_shieldKineticDamageResonance,dogma_shieldThermalDamageResonance,dogma_volume,is_buildable,blueprint_typeID,mfg_Isogen,mfg_Megacyte,mfg_Mexallon,mfg_Nocxium,mfg_Pyerite,mfg_Tritanium,mfg_Zydrine
0,582,Bantam,Frigate,Ship,Caldari,300000.0,0.5,0.9,225.0,0.75,0.55,615.0,200.0,0.0,500.0,1.0,0.5,0.6,0.8,0.0,True,683.0,375.0,0.0,1875.0,0.0,4500.0,24000.0,0.0
1,583,Condor,Frigate,Ship,Caldari,400000.0,0.5,0.9,250.0,0.75,0.55,300.0,185.0,0.0,400.0,1.0,0.5,0.6,0.8,0.0,True,684.0,500.0,0.0,2500.0,0.0,6000.0,32000.0,0.0
2,584,Griffin,Frigate,Ship,Caldari,300000.0,0.5,0.9,250.0,0.75,0.55,245.0,240.0,0.0,400.0,1.0,0.5,0.6,0.8,0.0,True,685.0,375.0,0.0,1875.0,0.0,4500.0,24000.0,0.0
3,585,Slasher,Frigate,Ship,Minmatar,400000.0,0.4,0.9,300.0,0.75,0.65,240.0,140.0,0.0,350.0,1.0,0.5,0.6,0.8,0.0,True,689.0,500.0,0.0,2500.0,0.0,6000.0,32000.0,0.0
4,586,Probe,Frigate,Ship,Minmatar,400000.0,0.4,0.9,300.0,0.75,0.65,235.0,240.0,0.0,300.0,1.0,0.5,0.6,0.8,0.0,True,690.0,550.0,0.0,2750.0,0.0,6600.0,35200.0,0.0


# Analyze Anomalies ("Design-Time" check)
It finds items that are statically imbalanced.

## Visualize Anomalies (EDA on the Master Vector)

In [ ]:
import plotly.express as px

# This cell assumes 'master_vector' is in memory

print("--- Visualizing Ship Features for Anomalies ---")

# Anomaly Plot 1: Cost vs. HP
# Hypothesis: More expensive ships (minerals) should have more HP.
fig = px.scatter(
    master_vector,
    x='dogma_armorHP',
    y='mfg_Tritanium',  # Tritanium is a good proxy for cost
    color='group_name', # See how Frigates, Cruisers, etc. cluster
    hover_name='ship_name',
    title='Anomaly Plot: Armor HP vs. Tritanium Cost',
    log_x=True, # Use log scale for HP
    log_y=True  # Use log scale for cost
)
fig.show()

# --- THIS PLOT IS NOW FIXED ---
# Anomaly Plot 2: CPU vs. Capacitor
# Hypothesis: Ships should have a balanced CPU and Capacitor.
fig2 = px.scatter(
    master_vector,
    x='dogma_cpuOutput',
    y='dogma_capacitorCapacity', # <-- CORRECTED: Was 'dogma_powergridOutput'
    color='group_name',
    hover_name='ship_name',
    title='Anomaly Plot: CPU vs. Capacitor'
)
fig2.show()


--- Visualizing Ship Features for Anomalies ---


## Build the Anomaly Detection Model (Unsupervised learning)

**Modeling (Scikit-learn):** We use an ***Isolation Forest*** model to "score" each ship based on how anomalous its features are.

In [ ]:
from sklearn.ensemble import IsolationForest
import numpy as np

# This cell assumes 'master_vector' is in memory

print("--- Building Anomaly Detection Model ---")

# 1. Select ONLY the numeric features for the model
# We can't model text, so we drop names, groups, etc.
features_to_model = [col for col in master_vector.columns if
                     col.startswith('dogma_') or col.startswith('mfg_')]

print(f"Modeling on {len(features_to_model)} features.")
X = master_vector[features_to_model]

# 2. Create and fit the model
# 'contamination' is your "guess" at what % of data is anomalous.
# Let's start by looking for the weirdest 1% (0.01)
model = IsolationForest(contamination=0.01, random_state=42)
model.fit(X)

# 3. Get the predictions
# The model gives a 'decision_function' score (lower is more anomalous)
# and a 'predict' score (-1 = anomaly, 1 = normal)
master_vector['anomaly_score'] = model.decision_function(X)
master_vector['is_anomaly'] = model.predict(X)

print("Model training complete. 'anomaly_score' and 'is_anomaly' added.")

# 4. Sort by the anomaly score to see the "weirdest" ships
df_anomalies = master_vector.sort_values(by='anomaly_score').reset_index()

print("\n--- Top 10 Most Anomalous Ships ---")
display(df_anomalies[df_anomalies['is_anomaly'] == -1][
    ['ship_name', 'group_name', 'anomaly_score']
].head(10))

--- Building Anomaly Detection Model ---
Modeling on 21 features.
Model training complete. 'anomaly_score' and 'is_anomaly' added.

--- Top 10 Most Anomalous Ships ---


,ship_name,group_name,anomaly_score
0,Cockroach,Frigate,-0.092350
1,Marshal,Black Ops,-0.063739
2,Monitor,Flag Cruiser,-0.044897
3,Thunderchild,Battleship,-0.022706
4,Nestor,Battleship,-0.004957


## Investigate a Top Anomaly

In [ ]:
# This cell assumes 'df_anomalies' is in memory

# Get the name of the #1 most anomalous ship
top_anomaly_name = df_anomalies.iloc[0]['ship_name']

print(f"--- Investigating Top Anomaly: {top_anomaly_name} ---")

# Get all features for that ship
anomaly_details = master_vector[
    master_vector['ship_name'] == top_anomaly_name
]

# Print all its stats
# .T (transpose) makes it easy to read
display(anomaly_details[features_to_model].T)

--- Investigating Top Anomaly: Cockroach ---


,112
dogma_armorEmDamageResonance,0.001
dogma_armorExplosiveDamageResonance,0.001
dogma_armorHP,1000000.000
dogma_armorKineticDamageResonance,0.001
dogma_armorThermalDamageResonance,0.001
dogma_capacitorCapacity,100000.000
dogma_cpuOutput,10000.000
dogma_mass,0.000
dogma_shieldCapacity,5000000.000
dogma_shieldEmDamageResonance,0.001


## "Game Health" Model (Filter for Player Ships)

In [ ]:
from sklearn.ensemble import IsolationForest
import numpy as np

# This cell assumes 'master_vector' is in memory

print("--- Building 'Game Health' Anomaly Model (Player Ships Only) ---")

# 1. Create a "clean" dataset: only ships that are buildable AND on the market
clean_vector = master_vector[
    (master_vector['is_buildable'] == True) &
    (master_vector['market_group'].notna())
].copy()

print(f"Original ship count: {len(master_vector)}. Filtered to {len(clean_vector)} buildable, marketable ships.")

# 2. Select features
features_to_model = [col for col in clean_vector.columns if
                     col.startswith('dogma_') or col.startswith('mfg_')]
X_clean = clean_vector[features_to_model]

# 3. Create and fit the new model
model_clean = IsolationForest(contamination=0.01, random_state=42)
model_clean.fit(X_clean)

# 4. Get predictions
clean_vector['anomaly_score'] = model_clean.decision_function(X_clean)
clean_vector['is_anomaly'] = model_clean.predict(X_clean)

# 5. Sort by the anomaly score
df_clean_anomalies = clean_vector.sort_values(by='anomaly_score').reset_index()

print("\n--- Top 10 'Player Ship' Anomalies ---")
display(df_clean_anomalies[df_clean_anomalies['is_anomaly'] == -1][
    ['ship_name', 'group_name', 'anomaly_score']
].head(10))

--- Building 'Game Health' Anomaly Model (Player Ships Only) ---
Original ship count: 550. Filtered to 385 buildable, marketable ships.

--- Top 10 'Player Ship' Anomalies ---


,ship_name,group_name,anomaly_score
0,Monitor,Flag Cruiser,-0.125228
1,Marshal,Black Ops,-0.101816
2,Thunderchild,Battleship,-0.047892
3,Leshak,Battleship,-0.005217


# Key Findings

**Model 1 (All Ships):** Successfully identified the "***Cockroach***" as the top anomaly—a "god-mode" admin frigate with millions of HP. This proves the model's ability to find statistical impossibilities.

**Model 2 (Player Ships):** After filtering out admin/unbuildable items, the model correctly identified rare, high-end ships like the "Marshal" and "Thunderchild" as the top outliers, proving it can find "edge cases" in the player economy.

# Save Feature Vectors

## Save Feature Vectors to GCS

In [ ]:
import pandas as pd

# This assumes 'master_vector' and 'clean_vector' are in memory
# and 'GCS_PATH' is defined.

print("--- Saving Feature Vectors to GCS ---")

# 1. Save the full 550-ship vector
try:
    master_vector.to_csv(f"{GCS_PATH}/all_ships_vector.csv", index=False)
    print(f"Successfully saved to: {GCS_PATH}/all_ships_vector.csv")

    # 2. Save the "clean" 385-ship vector for your game health model
    clean_vector.to_csv(f"{GCS_PATH}/player_ships_vector.csv", index=False)
    print(f"Successfully saved to: {GCS_PATH}/player_ships_vector.csv")

except Exception as e:
    print(f"*** FAILED to save vectors. Error: {e} ***")

--- Saving Feature Vectors to GCS ---
Successfully saved to: gs://eve-online-foundation-data/all_ships_vector.csv
Successfully saved to: gs://eve-online-foundation-data/player_ships_vector.csv


## Save Feature Vectors locally

In [ ]:
from google.colab import files
print("\n--- Downloading Master Vector Locally ---")
master_vector.to_csv('master_ship_vector.csv')
files.download('master_ship_vector.csv')
print("\n--- Downloading 385-ship vector for your game health model Locally ---")
clean_vector.to_csv('clean_player_ship_vector.csv')
files.download('clean_player_ship_vector.csv')


--- Downloading Master Vector Locally ---


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--- Save the clean 385-ship vector for your game health model Locally ---


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>